# 07 — Arquivos, JSON, CSV e exceções

**Objetivos:** manipular caminhos e arquivos com segurança, serializar dados e tratar falhas esperadas.


## Caminhos com `pathlib`

`Path` cria caminhos portáveis entre sistemas operacionais e oferece métodos para leitura, escrita e navegação.


In [1]:
from pathlib import Path

caminho = Path("dados") / "exemplo.txt"
print(caminho)
print(caminho.suffix, caminho.stem)


dados\exemplo.txt
.txt exemplo


## Gerenciadores de contexto

`with` garante o fechamento de recursos mesmo quando ocorre uma falha. Usaremos uma pasta temporária para que o notebook não deixe arquivos no projeto.


In [2]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as pasta_temporaria:
    arquivo = Path(pasta_temporaria) / "mensagem.txt"
    arquivo.write_text("Python\né legível\n", encoding="utf-8")
    conteudo = arquivo.read_text(encoding="utf-8")
    print(conteudo)


Python
é legível



## JSON

JSON representa objetos, listas, strings, números, booleanos e `null`. O módulo `json` converte entre JSON e estruturas Python.


In [3]:
import json

perfil = {"nome": "Ana", "ativo": True, "habilidades": ["Python", "SQL"]}
texto_json = json.dumps(perfil, ensure_ascii=False, indent=2)
recuperado = json.loads(texto_json)

print(texto_json)
print(recuperado["habilidades"])


{
  "nome": "Ana",
  "ativo": true,
  "habilidades": [
    "Python",
    "SQL"
  ]
}
['Python', 'SQL']


## CSV

O módulo `csv` lê e escreve dados tabulares. `DictReader` e `DictWriter` usam os nomes das colunas como chaves.


In [4]:
import csv
from io import StringIO

buffer = StringIO()
colunas = ["nome", "nota"]
escritor = csv.DictWriter(buffer, fieldnames=colunas)
escritor.writeheader()
escritor.writerows([
    {"nome": "Lia", "nota": 9.0},
    {"nome": "Caio", "nota": 7.5},
])

buffer.seek(0)
linhas = list(csv.DictReader(buffer))
print(buffer.getvalue())
print(linhas)


nome,nota
Lia,9.0
Caio,7.5

[{'nome': 'Lia', 'nota': '9.0'}, {'nome': 'Caio', 'nota': '7.5'}]


## Tratamento de exceções

Use `try` para a operação arriscada e capture exceções específicas com `except`. `else` roda em caso de sucesso e `finally` sempre roda.


In [5]:
entradas = ["42", "zero", "8"]
convertidos = []

for entrada in entradas:
    try:
        numero = int(entrada)
    except ValueError:
        print(f"Ignorando valor inválido: {entrada!r}")
    else:
        convertidos.append(numero)

print(convertidos)


Ignorando valor inválido: 'zero'
[42, 8]


## Lançando exceções

`raise` comunica que uma regra foi violada. Exceções personalizadas dão nomes claros aos erros do domínio.


In [6]:
class SaldoInsuficienteError(Exception):
    '''Indica que uma conta não possui saldo para a operação.'''


def sacar(saldo: float, valor: float) -> float:
    if valor <= 0:
        raise ValueError("O valor deve ser positivo")
    if valor > saldo:
        raise SaldoInsuficienteError("Saldo insuficiente")
    return saldo - valor


try:
    novo_saldo = sacar(100, 150)
except SaldoInsuficienteError as erro:
    print("Operação recusada:", erro)


Operação recusada: Saldo insuficiente


## Criando um gerenciador de contexto

`contextlib.contextmanager` permite encapsular preparação e limpeza ao redor de uma operação.


In [7]:
from contextlib import contextmanager

@contextmanager
def etapa(nome):
    print(f"Iniciando {nome}")
    try:
        yield
    finally:
        print(f"Finalizando {nome}")


with etapa("processamento"):
    print("Trabalhando...")


Iniciando processamento
Trabalhando...
Finalizando processamento


## Pratique

Leia uma configuração JSON, usando um valor padrão caso a chave `tema` não exista.


In [8]:
# Solução sugerida
configuracao_json = '{"idioma": "pt-BR", "notificacoes": true}'

try:
    configuracao = json.loads(configuracao_json)
except json.JSONDecodeError as erro:
    print("Configuração inválida:", erro)
    configuracao = {}

tema = configuracao.get("tema", "claro")
print("Tema:", tema)


Tema: claro


## Resumo

Use `Path` e `with` para trabalhar com arquivos, JSON/CSV para intercâmbio de dados e exceções específicas para falhas que o programa sabe tratar.
